# Baseline Rule-Label Model Guide

Purpose:
- train simple baseline classifiers for rule-based swing-high and swing-low labels
- produce reusable probability outputs and saved models for later detector experiments

Inputs:
- `ETHUSDT_15m_features_reduced.parquet`
- rule-label parquet referenced in the notebook

Outputs:
- baseline prediction parquet
- `model_rule_high_baseline.pkl`
- `model_rule_low_baseline.pkl`

Reading guide:
1. Load the feature matrix and rule labels.
2. Align both tables on `timestamp`.
3. Create a chronological train/test split.
4. Train one classifier for highs and one for lows.
5. Inspect and save the out-of-sample predictions and artifacts.

Important note:
- This notebook is a baseline benchmark, so the goal is fast comparability rather than the final production setup.


## 0) Optional environment setup


In [1]:
# Optional dependency install for a fresh notebook runtime.

!pip install pandas pyarrow fastparquet scikit-learn lightgbm


   ---------------------------------------- 0.0/669.2 kB ? eta -:--:--
   ---------------------------------------- 669.2/669.2 kB 3.3 MB/s  0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
    --------------------------------------- 0.3/15.8 MB ? eta -:--:--
    --------------------------------------- 0.3/15.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/15.8 MB 699.0 kB/s eta 0:00:22
   - -------------------------------------- 0.5/15.8 MB 699.0 kB/s eta 0:00:22
   - -------------------------------------- 0.5/15.8 MB 699.0 kB/s eta 0:00:22
   - -------------------------------------- 0.8/15.8 MB 524.3 kB/s eta 0:00:29
   - -------------------------------------- 0.8/15.8 MB 524.3 kB/s eta 0:00:29
   - -------------------------------------- 0.8/15.8 MB 524.3 kB/s eta 0:00:29
   -- ------------------------------------- 1.0/15.8 MB 488.8 kB/s eta 0:00:31
   -- -------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Path configuration


In [ ]:
from pathlib import Path

def locate_ml_root(start=None) -> Path:
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start] + list(start.parents):
        if candidate.name == "ML v1" and (candidate / "data").exists():
            return candidate

        ml_root = candidate / "ML v1"
        if (ml_root / "data").exists() and (ml_root / "code").exists():
            return ml_root

    raise FileNotFoundError(
        "Could not locate the 'ML v1' workspace from the current working directory."
    )


ML_ROOT = locate_ml_root()
PROJECT_ROOT = ML_ROOT.parent
DATA_DIR = ML_ROOT / "data"
CONFIG_DIR = ML_ROOT / "configs"
CODE_DIR = ML_ROOT / "code"

FEATURES_REDUCED_PATH = DATA_DIR / "ETHUSDT_15m_features_reduced.parquet"
RULE_LABELS_PATH = DATA_DIR / "swing_labels.parquet"
BASELINE_RULE_PREDICTIONS_PATH = DATA_DIR / "baseline_rule_predictions.parquet"
MODEL_RULE_HIGH_PATH = DATA_DIR / "model_rule_high_baseline.pkl"
MODEL_RULE_LOW_PATH = DATA_DIR / "model_rule_low_baseline.pkl"


## 1) Load features and rule labels


In [ ]:
# Load the reduced feature matrix and the rule-label parquet that will act as supervised targets.

import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# Load the centralized parquet files
features_df = pd.read_parquet(FEATURES_REDUCED_PATH)
labels_df = pd.read_parquet(RULE_LABELS_PATH)

print("Data loaded successfully!")
print("Labels columns available:", labels_df.columns.tolist())


## 2) Align both datasets on timestamp


In [9]:
# Normalize both tables onto the same timestamp index before joining them into one modeling dataset.

# Create safe copies to modify
features_tmp = features_df.copy()
labels_tmp = labels_df.copy()

# 1. Force both dataframes to use 'timestamp' as their index
if 'timestamp' in features_tmp.columns:
    features_tmp = features_tmp.set_index('timestamp')
    
if 'timestamp' in labels_tmp.columns:
    labels_tmp = labels_tmp.set_index('timestamp')

# 2. Convert both indexes to standard Pandas datetime formats
# This guarantees that strings like "2024-01-01" will perfectly match datetime objects!
features_tmp.index = pd.to_datetime(features_tmp.index)
labels_tmp.index = pd.to_datetime(labels_tmp.index)

# 3. Join them together
df = features_tmp.join(labels_tmp, how='inner')

# 4. Drop NA values and see the result
df = df.dropna()

print(f"Merged dataset shape: {df.shape}")


Merged dataset shape: (139219, 11)


## 3) Build `X` / `y` and create a chronological split


In [10]:
# Build separate high and low targets, then keep the split chronological to preserve time order.

target_high_col = 'y_high_rule' 
target_low_col = 'y_low_rule'

# Drop targets and metadata from the features (X)
columns_to_drop = [target_high_col, target_low_col]
if 'segment_id' in df.columns:
    columns_to_drop.append('segment_id')

X = df.drop(columns=columns_to_drop)
y_high = df[target_high_col]
y_low = df[target_low_col]

# Split into train and test chronologically (80% train, 20% test)
from sklearn.model_selection import train_test_split
X_train, X_test, y_high_train, y_high_test, y_low_train, y_low_test = train_test_split(
    X, y_high, y_low, test_size=0.2, shuffle=False
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")


Training set: 111375 rows
Test set: 27844 rows


## 4) Train separate baselines for swing highs and swing lows


In [11]:
# Train two lightweight baseline models so high and low swings can be evaluated independently.

from sklearn.metrics import roc_auc_score

def train_baseline(X_train, y_train, X_test, y_test, target_name):
    print(f"--- Training LightGBM for {target_name} ---")
    
    # Initialize LightGBM with balanced class weights 
    model = lgb.LGBMClassifier(
        n_estimators=100, 
        learning_rate=0.05, 
        class_weight='balanced', 
        random_state=42
    )
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Predict probabilities 
    probs = model.predict_proba(X_test)[:, 1]
    
    # Calculate ROC AUC score
    auc = roc_auc_score(y_test, probs)
    print(f"ROC AUC for {target_name}: {auc:.4f}\n")
    
    return probs, model

# Train to predict P(high)
p_high, model_high = train_baseline(X_train, y_high_train, X_test, y_high_test, "P(high)")

# Train to predict P(low)
p_low, model_low = train_baseline(X_train, y_low_train, X_test, y_low_test, "P(low)")


--- Training LightGBM for P(high) ---
[LightGBM] [Info] Number of positive: 13053, number of negative: 98322
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001023 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 111375, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
ROC AUC for P(high): 0.9138

--- Training LightGBM for P(low) ---
[LightGBM] [Info] Number of positive: 12677, number of negative: 98698
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002138 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 111375, number 

## 5) Inspect out-of-sample probabilities


In [12]:
# Attach out-of-sample probabilities back to the test slice for quick inspection.

# Attach predictions back to the test set to view them
results_df = X_test.copy()

results_df['actual_high'] = y_high_test
results_df['predicted_P_high'] = p_high
results_df['actual_low'] = y_low_test
results_df['predicted_P_low'] = p_low

# Display the top 15 rows of the results
display(results_df[['actual_high', 'predicted_P_high', 'actual_low', 'predicted_P_low']].head(15))


,actual_high,predicted_P_high,actual_low,predicted_P_low
timestamp,,,,
2024-09-11 19:45:00+00:00,0,0.771576,0,0.003310
2024-09-11 20:00:00+00:00,1,0.711402,0,0.003310
2024-09-11 20:15:00+00:00,0,0.812815,0,0.003310
2024-09-11 20:30:00+00:00,0,0.680281,0,0.003310
2024-09-11 20:45:00+00:00,1,0.705924,0,0.003310
2024-09-11 21:00:00+00:00,0,0.003310,0,0.694167
2024-09-11 21:15:00+00:00,0,0.003310,0,0.678091
2024-09-11 21:30:00+00:00,0,0.003309,0,0.003310
2024-09-11 21:45:00+00:00,0,0.003310,0,0.561371


## 6) Save predictions and fitted models


In [ ]:
# Persist both the predictions and the fitted models so the detector notebook can reuse them later.

import joblib

# 1. Save the final prediction results
results_df.to_parquet(BASELINE_RULE_PREDICTIONS_PATH)

# 2. Save the models so you don't have to retrain them later
joblib.dump(model_high, MODEL_RULE_HIGH_PATH)
joblib.dump(model_low, MODEL_RULE_LOW_PATH)

print("Models and predictions saved successfully!")
